<a href="https://colab.research.google.com/github/Danny3636/Generative-AI-Tasks/blob/main/Required_Task_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

From this task, I've learned how to successfully add a new function as a tool to a Gemini agent, specifically for fetching financial data like the P/E ratio. I also understand how the agent can automatically utilize these tools to answer complex questions, such as evaluating if a stock is overvalued based on a benchmark.

# Required Task 16 – Adding a P/E Ratio Tool to the Agent

This notebook extends the **Agents1** notebook by adding a new tool called `get_pe_ratio` that fetches the Price-to-Earnings ratio. We then ask the agent whether Apple is 'overvalued' compared to the average market P/E of 25.

### Setup and API Configuration

In [4]:
# 1. Setup
!pip install -q -U google-generativeai
!pip install -q yfinance

import google.generativeai as genai
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("✅ Setup complete.")

✅ Setup complete.


### Define All Tools (Including the New `get_pe_ratio`)

We keep the two original tools (`get_stock_price`, `get_company_risk_score`) and add the new `get_pe_ratio` tool that retrieves the trailing P/E ratio from Yahoo Finance.

In [3]:
import yfinance as yf

# --- Original Tools ---

def get_stock_price(ticker: str):
    """
    Retrieves the current live stock price for a given ticker.
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL', 'NVDA').
    """
    print(f"  ... 🔍 TOOL CALL: Connecting to Yahoo Finance for {ticker} ...")
    try:
        stock = yf.Ticker(ticker)
        price = stock.fast_info['last_price']
        return round(price, 2)
    except Exception as e:
        return f"Error fetching price for {ticker}: {e}"


def get_company_risk_score(ticker: str):
    """
    Calculates a risk proxy based on the stock's Beta (market volatility).
    A Beta > 1.0 means higher risk/volatility than the market.
    Args:
        ticker: The stock ticker symbol.
    """
    print(f"  ... ⚠️ TOOL CALL: Fetching Risk Metrics for {ticker} ...")
    try:
        stock = yf.Ticker(ticker)
        beta = stock.info.get('beta', 0)
        if beta > 1.5:
            assessment = "High Risk (High Volatility)"
        elif beta < 0.8:
            assessment = "Low Risk (Stable)"
        else:
            assessment = "Moderate Risk"
        return {"beta": beta, "assessment": assessment}
    except Exception as e:
        return "Risk data unavailable"


# --- NEW Tool for Task 16 ---

def get_pe_ratio(ticker: str):
    """
    Fetches the trailing Price-to-Earnings (P/E) ratio for a given stock.
    The P/E ratio compares a company's share price to its earnings per share.
    A higher P/E may suggest the stock is overvalued relative to its earnings,
    while a lower P/E may suggest it is undervalued.
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL', 'MSFT').
    """
    print(f"  ... 📊 TOOL CALL: Fetching P/E Ratio for {ticker} ...")
    try:
        stock = yf.Ticker(ticker)
        pe = stock.info.get('trailingPE', 'N/A')
        if pe != 'N/A':
            pe = round(pe, 2)
        return {"ticker": ticker, "trailing_pe": pe}
    except Exception as e:
        return f"Error fetching P/E ratio for {ticker}: {e}"


print("✅ All tools defined (including get_pe_ratio).")

✅ All tools defined (including get_pe_ratio).


### Initialize the Model with All Three Tools

In [5]:
# Initialize Model with all three tools
tools_list = [get_stock_price, get_company_risk_score, get_pe_ratio]

model = genai.GenerativeModel(
    "gemini-2.5-flash",
    tools=tools_list
)

chat = model.start_chat(enable_automatic_function_calling=True)

print("✅ Model initialized with all three tools (including P/E ratio).")

✅ Model initialized with all three tools (including P/E ratio).


### Ask the Agent: Is Apple Overvalued?

We prompt the agent to fetch Apple's P/E ratio and compare it against the average market P/E of 25. The agent should call `get_pe_ratio` with 'AAPL' and then reason about whether Apple appears overvalued.

In [6]:
query = (
    "Fetch Apple's (AAPL) P/E ratio using the tools available. "
    "The average market P/E ratio is roughly 25. "
    "Based on Apple's P/E compared to this benchmark, "
    "would you say Apple stock is overvalued, undervalued, or fairly valued? "
    "Explain your reasoning."
)

print(f"❓ USER QUERY: {query}\n")

response = chat.send_message(query)

print("\n🤖 AGENT RESPONSE:")
print(response.text)

❓ USER QUERY: Fetch Apple's (AAPL) P/E ratio using the tools available. The average market P/E ratio is roughly 25. Based on Apple's P/E compared to this benchmark, would you say Apple stock is overvalued, undervalued, or fairly valued? Explain your reasoning.

  ... 📊 TOOL CALL: Fetching P/E Ratio for AAPL ...

🤖 AGENT RESPONSE:
Apple's (AAPL) P/E ratio is 31.35. Compared to the average market P/E ratio of 25, Apple's P/E ratio is higher. This suggests that Apple stock may be overvalued relative to its earnings.


### Inspect the Agent's Thought Process

Let's look at the full chat history to see which tools the agent called and what observations it received before composing its final answer.

In [7]:
print("🕵️‍♂️ HISTORY INSPECTION:\n")
for message in chat.history:
    role = message.role
    print(f"--- {role.upper()} ---")
    for part in message.parts:
        if fn := part.function_call:
            print(f"🔧 ACTION: Calling Function '{fn.name}' with args: {dict(fn.args)}")
        elif resp := part.function_response:
            print(f"📨 OBSERVATION: Function returned: {resp.response}")
        else:
            print(part.text)

🕵️‍♂️ HISTORY INSPECTION:

--- USER ---
Fetch Apple's (AAPL) P/E ratio using the tools available. The average market P/E ratio is roughly 25. Based on Apple's P/E compared to this benchmark, would you say Apple stock is overvalued, undervalued, or fairly valued? Explain your reasoning.
--- MODEL ---
🔧 ACTION: Calling Function 'get_pe_ratio' with args: {'ticker': 'AAPL'}
--- USER ---
📨 OBSERVATION: Function returned: <proto.marshal.collections.maps.MapComposite object at 0x7a1afdb530b0>
--- MODEL ---
Apple's (AAPL) P/E ratio is 31.35. Compared to the average market P/E ratio of 25, Apple's P/E ratio is higher. This suggests that Apple stock may be overvalued relative to its earnings.
